# Chapter 06 Companion Notebook: Linear Regression

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch06_Linear_Regression.ipynb)

This notebook accompanies Chapter 06 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



<h1>Chapter 4. Linear Regression</h1>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
<a href="https://www.amazon.com/"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://github.com/"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>

---
- This notebook is a supplementary resource for the [Business Analytics and Artificial Intelligence](https://www.amazon.com/) book by [Aiden Lee](https://www.linkedin.com/in/ainmarketing/) and [Reo Song](https://www.linkedin.com/in/reo-song-57195a4b/).
- If you are using a cloud platform such as Google Colab, uncomment and run the following code block to install the dependencies for this chapter.
- We recommed to use a GPU  to run this notebook. In Google Colab, go to: Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4.
---

In [ ]:
# %%capture
# !pip install -q
# !pip install -q

# Movie: Linear regression

### Use “Movie.csv”
- title: movie title
- release: release date
- revenue: box office revenues
- budget: production budget
- rating: consumer ratings
- genres: movie geners
- country: production country

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as sm
# import statsmodels.api as sm  # alternative

In [ ]:
df = pd.read_csv('Movie.csv')
df.head()

### 1.Report summary statistics of 'revenue' and 'budget'.

In [ ]:
df.describe()

### 2.	Create dummy variables from 'country'

In [ ]:
# Creating dummy variables for 'country'
cntry_dum = pd.get_dummies(df.country, dtype=int)
  #  `dtype=int` ensures that the dummy variables are in numeric form (1 and 0).

# Adding the dummy variables to the original dataframe
df = pd.concat([df, cntry_dum], axis=1)
df.head()

### 3.Report summary statistics of Revenue by country.

In [ ]:
df.groupby('country').revenue.describe()

### 4.Run correlations on the following variables: 'revenue', 'budget', 'rating'. Is the correlation between Revenue and Rating statistically significant at 5% level?

In [ ]:
df[['revenue', 'budget', 'rating']].corr()

In [ ]:
stats.pearsonr(df.revenue, df.rating)  # corr and p-value

- The correlation between revenue and rating is statistically significant (p-value is less than 5%).

### 5. Draw the following scatterplot.
- Y-axis: Revenue, X-axis: Rating
- Use different colors for different countries.  

In [ ]:
# Creating the scatter plot with different colors for different countries
plt.figure(figsize=(10, 6))
for country in df['country'].unique():
    subset = df[df['country'] == country]
    plt.scatter(subset['rating'], subset['revenue'], label=country)

# Adding labels, legend, and title
plt.xlabel('Rating')
plt.ylabel('Revenue')
plt.title('Scatterplot of Revenue vs. Rating')
plt.legend(title='Country')

- Rating and revenue have a positive and potentially non-linear relationship.

### 6. Run the following regression. What is R2 and adjusted R2?
* Revenue = b0 + b1 Budget + b2 Rating + b3 US + b4 UK + e (e: error term)

In [ ]:
# Method 1

import statsmodels.formula.api as sm

m = sm.ols('revenue ~ budget + rating + US + UK', df).fit()  # constant is added by default
m.summary()

In [ ]:
m.params  # Get only coefficients

In [ ]:
# Method 2

import statsmodels.api as sm1

m1 = sm1.OLS.from_formula('revenue ~ budget + rating + US', df).fit()  # constant is added by default
m1.summary()

In [ ]:
# Method 3

import statsmodels.api as sm1

y=df.revenue
x=df[['budget', 'rating', 'US', 'UK']]
x=sm1.add_constant(x)  # Need to add constant to x

m2 = sm1.OLS(y, x).fit()
m2.summary()

### 7. Which independent variables are significant? (Use 5% significance level)

* All variables are significant at 1% level (p-values are less than 1%)

###  8.	Interpret statistically significant coefficients. (Use 5% significance level)

* A one dollar increase in budget increases box office revenues by 3.17 dollar.
* A one point increase in rating increases box office revenues by 43.11 million dollar.
* US movies earn 27.57 million dollar more in box office revenues than than France movies.
* UK movies earn 30.91 million dollar more in box office revenues than than France movies.

### 9 Predict revenue for the following movie.
- Budget=$\$$10 million, Rating=7, US movies

In [ ]:
m.predict({'budget':10000000, 'rating':7, 'US':1, 'UK':0})

- The revenue of the movie will be $58.74 millions.

### 10.Generate predicted revenues for the all observations in the data.

In [ ]:
pred_rev=m.predict(df)
pred_rev.head()

In [ ]:
pred_rev=pd.DataFrame(m.predict(df), columns=['pred_rev'])  # Change to dataframe
pred_rev.head()

In [ ]:
df = pd.concat([df, pred_rev], axis=1)
df.head()